# Beta-Binomial Posterior Update

We have a coin with unknown bias θ. We'll flip it 200 times and, at each step,
update our belief about θ using Bayes' rule.

**The setup:** our belief is a Beta distribution. One head shifts it right;
one tail shifts it left. After enough flips, the data overwhelms the prior.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from scipy.stats import beta as beta_dist

In [ ]:
# Style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# Color palette
C_PRIOR     = '#4C72B0'
C_POST      = '#DD8452'
C_TRUE      = '#55A868'
TRUE_THETA  = 0.7

# Shared theta grid (avoids singularity at 0 and 1)
THETA = np.linspace(0.001, 0.999, 500)

# Shared dataset — seeded for reproducibility; Act 3 "New dataset" button resamples
rng  = np.random.default_rng(42)
data = rng.binomial(1, TRUE_THETA, 200)

## Act 1: Your Prior Belief

Before seeing any data, we commit to a starting belief about θ.

A **Beta(α, β)** distribution encodes this:
- **α** acts like "prior heads" — higher α pulls the curve toward 1
- **β** acts like "prior tails" — higher β pulls the curve toward 0
- **Effective sample size = α + β** — controls how confident you are

Drag the sliders to see how the shape changes.

In [ ]:
# --- Shared widgets (reused in Act 3) ---
alpha_slider = widgets.FloatSlider(
    value=3.0, min=0.5, max=15.0, step=0.5,
    description='α  (prior heads)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='380px')
)
beta_slider = widgets.FloatSlider(
    value=2.0, min=0.5, max=15.0, step=0.5,
    description='β  (prior tails)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='380px')
)

prior_out = widgets.Output()

def _draw_prior(_=None):
    a, b = alpha_slider.value, beta_slider.value
    pdf  = beta_dist.pdf(THETA, a, b)
    mean = a / (a + b)
    ess  = a + b
    with prior_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(THETA, pdf, color=C_PRIOR, linewidth=2.5)
        ax.fill_between(THETA, pdf, alpha=0.2, color=C_PRIOR)
        ax.axvline(mean, color=C_PRIOR, linestyle='--', linewidth=1.5, alpha=0.8,
                   label=f'Prior mean = {mean:.2f}')
        ax.set_xlabel('Probability θ could be the true coin bias')
        ax.set_ylabel('Belief density')
        ax.set_title(f'Prior  ·  Mean: {mean:.2f}  ·  Effective sample size: {ess:.0f}',
                     fontsize=12)
        ax.set_xlim(0, 1)
        ax.legend(fontsize=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        display(fig)
        plt.close(fig)

alpha_slider.observe(_draw_prior, names='value')
beta_slider.observe(_draw_prior, names='value')

display(widgets.VBox([
    widgets.HTML('<b>Shape your prior belief about the coin bias:</b>'),
    widgets.HBox([alpha_slider, beta_slider]),
    prior_out,
]))
_draw_prior()

## Act 2: One Observation

What happens after a single flip? The prior shifts — slightly — toward the observed outcome.

The update rule is exact:
- After a **Head**: Beta(α, β) → **Beta(α + 1, β)**
- After a **Tail**: Beta(α, β) → **Beta(α, β + 1)**

In [ ]:
def _one_observation_plot():
    a, b = alpha_slider.value, beta_slider.value
    prior_pdf = beta_dist.pdf(THETA, a, b)
    post_H    = beta_dist.pdf(THETA, a + 1, b)
    post_T    = beta_dist.pdf(THETA, a,     b + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, post_pdf, result in zip(axes, [post_H, post_T], ['Head (1)', 'Tail (0)']):
        ax.plot(THETA, prior_pdf, color=C_PRIOR, linewidth=2,
                linestyle='--', alpha=0.7, label='Prior')
        ax.fill_between(THETA, prior_pdf, alpha=0.12, color=C_PRIOR)
        ax.plot(THETA, post_pdf, color=C_POST, linewidth=2.5, label='Posterior')
        ax.fill_between(THETA, post_pdf, alpha=0.30, color=C_POST)
        ax.axvline(TRUE_THETA, color=C_TRUE, linestyle='--',
                   linewidth=1.5, label=f'True θ = {TRUE_THETA}')
        ax.set_title(f'After 1 flip: {result}', fontsize=12)
        ax.set_xlabel('Probability θ could be the true coin bias')
        ax.set_xlim(0, 1)
        ax.legend(fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    axes[0].set_ylabel('Belief density')
    plt.suptitle('One observation — how much does the posterior shift?',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

_one_observation_plot()

## Act 3: Learning in Action

Watch the posterior update across 200 coin flips. The true bias is **θ = 0.7**.

Use the α/β sliders from Act 1 to set your prior, then hit **▶ Replay**.
Try a very wrong prior (e.g. α=1, β=9) — does the posterior still find the truth?

In [ ]:
speed_widget = widgets.ToggleButtons(
    options=['Slow', 'Medium', 'Fast'],
    value='Medium',
    description='Speed:',
    layout=widgets.Layout(width='auto'),
)
replay_btn  = widgets.Button(description='▶ Replay',      button_style='primary')
newdata_btn = widgets.Button(description='🎲 New dataset', button_style='warning')
anim_out    = widgets.Output()

_current_data = {'d': data.copy()}   # mutable container so closures share state
SPEED_MAP = {'Slow': 120, 'Medium': 50, 'Fast': 12}

def _build_animation(d):
    """Return a FuncAnimation over 100 frames (every 2nd of 200 flips)."""
    a, b = alpha_slider.value, beta_slider.value
    prior_pdf = beta_dist.pdf(THETA, a, b)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.set_xlim(0, 1)

    # Static reference elements drawn once
    ax.plot(THETA, prior_pdf, color=C_PRIOR, linewidth=1.5,
            linestyle='--', alpha=0.6, label='Prior (fixed)')
    ax.fill_between(THETA, prior_pdf, alpha=0.10, color=C_PRIOR)
    ax.axvline(TRUE_THETA, color=C_TRUE, linestyle='--',
               linewidth=1.5, label=f'True θ = {TRUE_THETA}')

    post_line, = ax.plot([], [], color=C_POST, linewidth=2.5, label='Posterior')
    ax.set_xlabel('Probability θ could be the true coin bias')
    ax.set_ylabel('Belief density')
    title_obj = ax.set_title('Flips seen: 0')
    ax.legend(loc='upper left', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    fills = []   # track fill_between patches so we can remove them each frame

    def _update(frame):
        for f in fills:
            f.remove()
        fills.clear()
        n     = 2 * frame + 1      # every-other observation (1, 3, 5, … 199)
        heads = int(d[:n].sum())
        post_pdf = beta_dist.pdf(THETA, a + heads, b + (n - heads))
        post_line.set_data(THETA, post_pdf)
        fills.append(ax.fill_between(THETA, post_pdf, alpha=0.30, color=C_POST))
        post_mean = (a + heads) / (a + b + n)
        ax.relim()
        ax.autoscale_view(scalex=False)
        title_obj.set_text(
            f'Flips seen: {n}  ·  Posterior mean: {post_mean:.3f}'
        )
        return post_line,

    anim = FuncAnimation(
        fig, _update,
        frames=100,                        # 100 frames = every 2nd of 200 flips
        interval=SPEED_MAP[speed_widget.value],
        blit=False,
    )
    plt.close(fig)
    return anim

def _play(_=None):
    with anim_out:
        clear_output(wait=True)
        print('Generating animation — please wait…')
    anim = _build_animation(_current_data['d'])
    with anim_out:
        clear_output(wait=True)
        display(HTML(anim.to_jshtml()))

def _new_data(_=None):
    _current_data['d'] = np.random.binomial(1, TRUE_THETA, 200)
    _play()

replay_btn.on_click(_play)
newdata_btn.on_click(_new_data)

display(widgets.VBox([
    widgets.HTML(
        '<b>Set your prior above, then click Replay. '
        'Change Speed before clicking Replay to take effect.</b>'
    ),
    widgets.HBox([speed_widget, replay_btn, newdata_btn]),
    anim_out,
]))
_play()

## Act 4: Does the Prior Matter?

Three students start with very different priors. After enough flips, do they
all arrive at the same answer?

- **Uniform Beta(1, 1):** no prior knowledge — all biases equally plausible
- **Correct Beta(7, 3):** already suspects the coin is biased toward heads
- **Wrong Beta(2, 8):** wrongly suspects the coin is biased toward tails

In [ ]:
def _prior_comparison():
    priors = [
        ('Uniform  Beta(1,1)',  1.0, 1.0),
        ('Correct  Beta(7,3)',  7.0, 3.0),
        ('Wrong    Beta(2,8)',  2.0, 8.0),
    ]
    flip_counts = [10, 50, 200]

    fig, axes = plt.subplots(3, 3, figsize=(13, 10), sharex=True)

    for row, (label, a, b) in enumerate(priors):
        prior_pdf = beta_dist.pdf(THETA, a, b)
        for col, n in enumerate(flip_counts):
            ax    = axes[row, col]
            heads = int(data[:n].sum())
            post_pdf   = beta_dist.pdf(THETA, a + heads, b + (n - heads))
            post_mean  = (a + heads) / (a + b + n)

            ax.plot(THETA, prior_pdf, color=C_PRIOR, linewidth=1.5,
                    linestyle='--', alpha=0.7)
            ax.fill_between(THETA, prior_pdf, alpha=0.12, color=C_PRIOR)
            ax.plot(THETA, post_pdf, color=C_POST, linewidth=2)
            ax.fill_between(THETA, post_pdf, alpha=0.30, color=C_POST)
            ax.axvline(TRUE_THETA, color=C_TRUE, linestyle='--',
                       linewidth=1.2, alpha=0.9)
            ax.annotate(f'mean = {post_mean:.2f}', xy=(0.05, 0.91),
                        xycoords='axes fraction', fontsize=8, color=C_POST,
                        fontweight='bold')
            ax.set_xlim(0, 1)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            if col == 0:
                ax.set_ylabel(label, fontsize=9, labelpad=8)
            if row == 0:
                ax.set_title(f'After {n} flips', fontsize=10)
            if row == 2:
                ax.set_xlabel('θ', fontsize=9)

    # Shared legend
    prior_patch = mpatches.Patch(color=C_PRIOR, alpha=0.5, label='Prior')
    post_patch  = mpatches.Patch(color=C_POST,  alpha=0.5, label='Posterior')
    true_line   = plt.Line2D([0], [0], color=C_TRUE, linestyle='--',
                              label=f'True θ = {TRUE_THETA}')
    fig.legend(
        handles=[prior_patch, post_patch, true_line],
        loc='lower center', ncol=3, fontsize=10,
        bbox_to_anchor=(0.5, -0.02),
    )
    fig.suptitle('Does the prior matter? All roads lead to θ = 0.7',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

_prior_comparison()